In [ ]:
# Revisiting the "most" important features -- this time using regularisation methods.
#
# Lasso, Ridge, and ElasticNet regression models.

In [ ]:
%%python -m pip install --upgrade pip
%pip install pandas scikit-learn matplotlib

In [ ]:
from pandas import DataFrame, read_csv
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Load some data for regression

dataset = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=0)

In [ ]:
# Problem -- many variables seem related to the outcome. Which features are the "best" ones?

df = DataFrame(data=dataset.data, columns=dataset.feature_names)
df['y'] = dataset.target

corr = abs(df.corr())
corr.style.background_gradient(cmap='coolwarm').format(precision=2)

In [ ]:
# Baseline model 1 -- linear regression

pipe_regression = Pipeline([
  ('std', RobustScaler()),
  ('linreg', LinearRegression())
])

print("Score: %.3f" % pipe_regression.fit(X_train, y_train).score(X_test, y_test))
y_pred = pipe_regression.predict(X_test)
print("RMSE: %.3f" % root_mean_squared_error(y_test, y_pred))

DataFrame(abs(pipe_regression['linreg'].coef_), index=dataset.feature_names, columns=['coef'])


In [ ]:
# Lasso model -- regularisation with L1 penalty

pipe_lasso = Pipeline([
  ('std', RobustScaler()),
  ('lasso', LassoCV(cv=10))
])

print("Score: %.3f" % pipe_lasso.fit(X_train, y_train).score(X_test, y_test))
y_pred = pipe_lasso.predict(X_test)
print("RMSE: %.3f" % root_mean_squared_error(y_test, y_pred))

# Best alpha is in: pipe_lasso['lasso'].alpha_
DataFrame(abs(pipe_lasso['lasso'].coef_), index=dataset.feature_names, columns=['coef'])

In [ ]:
# Ridge model -- regularisation with L2 penalty

pipe_ridge = Pipeline([
  ('std', RobustScaler()),
  ('ridge', RidgeCV(cv=20))
])

print("Score: %.3f" % pipe_ridge.fit(X_train, y_train).score(X_test, y_test))
y_pred = pipe_ridge.predict(X_test)
print("RMSE: %.3f" % root_mean_squared_error(y_test, y_pred))

# Best alpha is in: pipe_ridge['ridge'].alpha_
DataFrame(abs(pipe_ridge['ridge'].coef_), index=dataset.feature_names, columns=['coef'])

In [ ]:
# ElasticNet model -- regularisation with L1 and L2 penalty terms

pipe_eNet = Pipeline([
  ('std', RobustScaler()),
  ('eNet', ElasticNetCV(cv=20))
])

print("Score: %.3f" % pipe_eNet.fit(X_train, y_train).score(X_test, y_test))
y_pred = pipe_eNet.predict(X_test)
print("RMSE: %.3f" % root_mean_squared_error(y_test, y_pred))

# Best alpha is in: pipe_eNet['eNet'].alpha_
DataFrame(abs(pipe_eNet['eNet'].coef_), index=dataset.feature_names, columns=['coef'])

In [ ]:
# We did not find much coefficients being zero ...
# How does it compare to permutation importance?

# Baseline model 2 -- random forest regression

pipe_rf = Pipeline([
  ('std', RobustScaler()),
  ('rf', RandomForestRegressor())
])

print("Score: %.3f" % pipe_rf.fit(X_train, y_train).score(X_test, y_test))
y_pred = pipe_rf.predict(X_test)
print("RMSE: %.3f" % root_mean_squared_error(y_test, y_pred))

In [ ]:
# https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance.html
from sklearn.inspection import permutation_importance

result = permutation_importance(
    pipe_rf, X_test, y_test, n_repeats=10, random_state=0, n_jobs=2
)

sorted_importances_idx = result.importances_mean.argsort()
importances = DataFrame(
    result.importances[sorted_importances_idx].T,
    columns=df.columns[sorted_importances_idx],
)

ax = importances.plot.box(vert=False, whis=10)
ax.set_title("Permutation Importances (test set)")
ax.axvline(x=0, color="k", linestyle="--")
ax.set_xlabel("Decrease in accuracy score")
ax.figure.tight_layout()

In [ ]:
# let's try a reduced model

X_train, X_test, y_train, y_test = train_test_split(
    df[['s5', 'bmi', 'bp', 'age', 's1']],
    df['y'], test_size=0.2, random_state=0)

print("Score: %.3f" % pipe_rf.fit(X_train, y_train).score(X_test, y_test))
y_pred = pipe_rf.predict(X_test)
print("RMSE: %.3f" % root_mean_squared_error(y_test, y_pred))
